# 🚀 Maya-1: Google Colab Ücretsiz T4 GPU ile ORPO Hizalama Eğitimi

Bu notebook, **Maya-1 1.1B** dil modelinin Türkçe sohbet yeteneğini (ORPO Hizalaması) Google Colab'in sunduğu **ücretsiz Nvidia T4 GPU** üzerinde tamamlamak için tasarlanmıştır.

### 📌 Başlamadan Önce:
Yukarıdaki menüden **Çalışma Zamanı ➡️ Çalışma zamanı türünü değiştir** seçeneğine tıklayın ve donanım hızlandırıcı olarak **T4 GPU** seçildiğinden emin olun.

In [ ]:
# GPU Durumunu ve Modeli Kontrol Et
!nvidia-smi

## 🏗️ 1. Projeyi Klonlayın ve Klasöre Geçin

In [ ]:
!git clone https://github.com/Abdullah6262637/Maya-1.git
%cd Maya-1

## 📦 2. Gerekli Kütüphaneleri Yükleyin

In [ ]:
!pip install -r requirements.txt
!pip install numpy==1.26.4 model2vec mup pandas joblib

## 🧹 3. 100K Kaliteli Türkçe Hizalama Verisini Süzün

Hugging Face üzerindeki Türkçe talimat verilerini indirip, projedeki yerel model2vec kalite modelimizle süzerek `shared/chat_data_orpo_100k.jsonl` dosyasını Colab içinde oluşturuyoruz:

In [ ]:
!python scripts/curate_alpaca_quality.py \
    --mode filter-dataset \
    --classifier_path shared/quality_classifier_test.bin \
    --output_path shared/chat_data_orpo_100k.jsonl

## 💾 4. Google Drive'ı Bağlayın (Önerilir)

Eğitilen model ağırlıklarının (checkpoint) Colab oturumu kapandığında kaybolmaması için Google Drive'ınızı bağlamanızı öneririz:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Google Drive üzerinde kayıt klasörünü hazırlayalım
!mkdir -p "/content/drive/MyDrive/maya_checkpoints"

## ⚡ 5. ORPO Hizalama Eğitimini Başlatın

In [ ]:
!python python_training/orpo_train.py \
    --data_path shared/chat_data_orpo_100k.jsonl \
    --resume_from shared/checkpoints/ckpt_latest.pt \
    --epochs 1 \
    --batch_size 2 \
    --seq_len 512 \
    --use_mup \
    --mup_base_hidden 64

## 📤 6. Eğitilen Modeli Google Drive'a Kaydedin

Eğitim bittiğinde ortaya çıkan `sft_checkpoint.pt` dosyasını kalıcı olarak Google Drive'ınıza kopyalayın:

In [ ]:
!cp -v shared/checkpoints/sft_checkpoint.pt "/content/drive/MyDrive/maya_checkpoints/sft_checkpoint.pt"